# **Fourth Homework Assignment** — Lenia with MPI

In [2]:
import numpy as np
import pandas as pd
import re

# Used for setting display settings to increase readability.
from IPython.core.display import display_html

In [3]:
# Custom formater.
def fromat_nan(val):
    if pd.isna(val):
        return ""
    if isinstance(val, float):
        return f"{round(val, 3)}"
    return f"{val}"

# Helper function that displays each dataframe in its own column.
def display_list(dfs, index=True, axis=-1, minmax="min"):
    # Convert each split to HTML with proper styling for side-by-side display.
    html_str = "<table><tr>" 
    for df in dfs:
        # Dataframe styling options.
        styled_df = df.style.format(fromat_nan)
        if axis >= 0 and minmax == "min":
            styled_df = styled_df.highlight_min(axis=axis, color='darkgreen')
            styled_df = styled_df.highlight_max(axis=axis, color='darkred')
        if axis >= 0 and minmax == "max":
            styled_df = styled_df.highlight_max(axis=axis, color='darkgreen')
            styled_df = styled_df.highlight_min(axis=axis, color='darkred')
        styled_html = styled_df.to_html(index=index)

        # Enables newlines and bold titles.
        name_html = df.attrs['name'].replace('\n', '<br>')
        name_html = f"<div style='text-align: left; font-weight: bold;'>{name_html}</div>"
        html_str += f"<td style='vertical-align: top; padding: 5px;'>{name_html + styled_html}</td>"
    html_str += "</tr></table>"
    display_html(html_str, raw=True)
    
# Helper function to display one data frame as multiple columns.
def display(df, cols, index=True):
    # Split the dataframe into columns.
    ratio = int(np.ceil(len(df) / cols))
    df_split = [df.iloc[i * ratio: (i + 1) * ratio] for i in range(cols)]
    display_list(df_split, index)


In [4]:
# Calculates means based on axis and returns the dataframe.
def get_means(dfs, axis):
    lst = []
    for df in dfs:
        mean = df.mean(axis).to_frame(name='Mean')
        mean.attrs['name'] = df.attrs['name']
        lst.append(mean)
    return lst

## **Time Comparisons**

In [ ]:
import glob
from pathlib import Path

# CSV columns produced by benchmarks/bench.sh: Run,Size,Method,Procs,Nodes,Halo,Time
_csv = pd.concat([pd.read_csv(f) for f in glob.glob("results_*.csv")], ignore_index=True)
_csv.columns = [c.strip() for c in _csv.columns]
_csv["Time"] = pd.to_numeric(_csv["Time"], errors="coerce")
for c in ("Size", "Procs", "Nodes", "Halo"):
    _csv[c] = pd.to_numeric(_csv[c], errors="coerce").astype("Int64")
_csv = _csv.dropna(subset=["Time"])

# Mean across runs for every (Size, Method, Procs, Nodes, Halo) cell.
_means = _csv.groupby(["Size", "Method", "Procs", "Nodes", "Halo"])["Time"].mean()

sizes = [128, 512, 1024, 2048, 4096]
procs = [1, 2, 4, 16, 32]

In [ ]:
def _t(method, size, p, nodes=1, halo=1):
    key = (size, method, p, nodes, halo)
    return _means[key] if key in _means.index else np.nan

# Average execution time per (Size, Procs) on a single node, halo=1.
def _time_table(method):
    df = pd.DataFrame(index=sizes, columns=procs, dtype=float)
    df.index.name = "Size"
    df.columns.name = "Procs"
    for s in sizes:
        for p in procs:
            df.loc[s, p] = _t(method, s, p)
    df.attrs["name"] = f"Average execution time (s), method = {method}"
    return df

print("Maximal values per row are displayed in red, while minimal are displayed in green.")
display_list([_time_table("row"), _time_table("block")], axis=1)

In [ ]:
# Speed-up S = t_s / t_p(P) where t_s is the sequential baseline.
def _speedup_table(method):
    df = pd.DataFrame(index=sizes, columns=procs, dtype=float)
    df.index.name = "Size"
    df.columns.name = "Procs"
    for s in sizes:
        ts = _t("seq", s, 1)
        for p in procs:
            tp = _t(method, s, p)
            df.loc[s, p] = (ts / tp) if (tp and tp > 0) else np.nan
    df.attrs["name"] = f"Speed-up t_s / t_p, method = {method}"
    return df

print("Maximal values per row are displayed in green, while minimal are displayed in red.")
display_list([_speedup_table("row"), _speedup_table("block")], axis=1, minmax="max")

In [ ]:
# 1-node vs 2-nodes for the same total process count (bonus).
def _nodes_table(method):
    rows = []
    for s in sizes:
        for p in procs:
            t1 = _t(method, s, p, nodes=1)
            t2 = _t(method, s, p, nodes=2)
            if pd.notna(t1) or pd.notna(t2):
                rows.append((s, p, t1, t2, (t1 - t2) if (pd.notna(t1) and pd.notna(t2)) else np.nan))
    df = pd.DataFrame(rows, columns=["Size", "Procs", "1 node (s)", "2 nodes (s)", "delta (s)"])
    df.attrs["name"] = f"1-node vs 2-nodes, method = {method}"
    return df

display_list([_nodes_table("row"), _nodes_table("block")])

In [ ]:
# Wide-halo K sweep (bonus). Halo width = K * R, exchange every K iterations.
_wide = _csv[_csv["Method"] == "row_wide"].copy()
if not _wide.empty:
    df_wide = _wide.groupby(["Size", "Procs", "Halo"])["Time"].mean().unstack("Halo")
    df_wide.attrs["name"] = "Wide-halo (lenia_row_wide): mean time per K, indexed by (Size, Procs)"
    display_list([df_wide], axis=1)
else:
    print("No row_wide results found yet.")